## 데이터 수집 자동화

### 다나와 상품 정보 스크래핑

#### 자동 로그인
- 패스워드 숨기기
- setx 명령어로 시스템에 저장 방법
##### 1. setx 명령어로 시스템에 저장방법
- powershell

```powershell
> setx DANAWA_ID "아이디"
성공 : 지정한 값을 저장했습니다.

> setx DANAWA_PASSWORD "비밀번호"

성공 : 지정한 값을 저장했습니다.
```
##### 2. keyring 으로 윈도우 자격증명 저장소
- setx 와 달리 아예 오픈 안됨
- 패키지 설치

In [14]:
import sys
import importlib.util

print("노트북 Python:", sys.executable)
print("keyring 모듈:", importlib.util.find_spec("keyring"))

노트북 Python: c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\python.exe
keyring 모듈: ModuleSpec(name='keyring', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000001FEDDBD1DF0>, origin='c:\\Users\\User\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\keyring\\__init__.py', submodule_search_locations=['c:\\Users\\User\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\keyring'])


In [15]:
# !pip install keyring

In [16]:
%pip install keyring

Note: you may need to restart the kernel to use updated packages.


In [17]:
import keyring
from getpass import getpass

In [38]:
username = input('다나와 아이디')
password = getpass('다나와 비밀번호')

# 아이디 저장
keyring.set_password('danawa.com', 'user_id', username)
# 패스워드
keyring.set_password('danawa.com', username, password)

print('아이디/패스워드 저장 완료')

아이디/패스워드 저장 완료


In [40]:
user_id = keyring.get_password('danawa.com', 'user_id')
user_id

'personar'

In [41]:
password = keyring.get_password('danawa.com', user_id)
password is not None

True

In [35]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [22]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.danawa.com/')

wait.until(
    EC.presence_of_all_elements_located((By.ID, 'danawa_footer'))
)

[<selenium.webdriver.remote.webelement.WebElement (session="efd63649932ac9654d61046724ada38a", element="f.D1914C456BE6ABE4787AB21172C3210C.d.AE8B473732E12F328E6E047232611C0D.e.77")>]

In [23]:
# 로그인 링크 확인 후 클릭
driver.find_element(By.LINK_TEXT, '로그인').click()

In [30]:
user_id

'haneulnuna'

In [31]:
password is not None

False

In [42]:
# 로그인 하기
id_input = driver.find_element(By.CSS_SELECTOR, 'input[type="text"]')
id_input.send_keys(user_id)

pass_input = driver.find_element(By.CSS_SELECTOR, 'input[type="password"]')
pass_input.send_keys(password)

In [43]:
login_btn = driver.find_element(By.CLASS_NAME, 'btn_login')
login_btn.click()

#### 동적 페이지 스크래핑

- https://prod.danawa.com/list/?cate=11255834&15main_11_02
- 90개로 조정 후 시작 

In [44]:
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    EC.presence_of_all_elements_located((By.TAG_NAME, 'footer'))
)

[<selenium.webdriver.remote.webelement.WebElement (session="efd63649932ac9654d61046724ada38a", element="f.D1914C456BE6ABE4787AB21172C3210C.d.8520BB399C44AA41AC259A9F79DFE8F5.e.10061")>]

- select 태그 페이지 내에 1개만 존재

In [51]:
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
           option.click()

wait.until(
EC.presence_of_all_elements_located((By.TAG_NAME, 'footer'))
)

[<selenium.webdriver.remote.webelement.WebElement (session="efd63649932ac9654d61046724ada38a", element="f.D1914C456BE6ABE4787AB21172C3210C.d.8520BB399C44AA41AC259A9F79DFE8F5.e.10061")>]

In [54]:
products = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]')
len(products)

90

In [55]:
product = products[0]
product.text

'1\n영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060\n5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / AMD / 라이젠 5000시리즈 / 라이젠5 / 버미어 / DDR4 / 16GB / M.2 / 256GB / NVIDIA / 그래픽 메모리: 8GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / LED쿨러 / 용도: 게임용\n혜택 최저가\n1,125,110원\n[옥션] 삼성카드\n/\n무이자 최대 24개월\n기획전\n성능도 가격도 GOOD! 조립PC 인기상품 추천\n21.03. 등록\n브랜드로그\n의견 70\n4.8\n리뷰수\n(230)\n관심\n대량구매\n2위\n16GB, M.2 256GB\n1,209,640\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.\n16GB, M.2 512GB\n1,259,670\n원\n5몰\nVS상품비교에 추가\n1위\n32GB, M.2 1TB\n1,539,640\n원\n5몰\nVS상품비교에 추가'

In [62]:
computer_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[1]
computer_info.text

'1\n영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060\n5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / AMD / 라이젠 5000시리즈 / 라이젠5 / 버미어 / DDR4 / 16GB / M.2 / 256GB / NVIDIA / 그래픽 메모리: 8GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / LED쿨러 / 용도: 게임용\n혜택 최저가\n1,125,110원\n[옥션] 삼성카드\n/\n무이자 최대 24개월\n기획전\n성능도 가격도 GOOD! 조립PC 인기상품 추천\n21.03. 등록\n브랜드로그\n의견 70\n4.8\n리뷰수\n(230)\n관심\n대량구매'

In [63]:
# 컴퓨터 이름 추출!!
computer_info.find_element(By.CSS_SELECTOR, 'div a').text

'영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060'

In [73]:
# 컴퓨터 사양
specs = computer_info.find_elements(By.CSS_SELECTOR,'div[data-testid="productListSpecs"]')

'\n'.join(spec.text for spec in specs)

''

In [69]:
price_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[2]
price_info.text

'2위\n16GB, M.2 256GB\n1,209,640\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.\n16GB, M.2 512GB\n1,259,670\n원\n5몰\nVS상품비교에 추가\n1위\n32GB, M.2 1TB\n1,539,640\n원\n5몰\nVS상품비교에 추가'

In [81]:
# 가격정보 추출
price_info.find_element(By.CSS_SELECTOR, 'li').text

AttributeError: 'str' object has no attribute 'find_element'